# **Day 62: Regular Expressions (Part 1)**

A **regular expression** (called **regex**) is a special pattern used to **search** for text.

**Example:** "Find all 6-digit numbers in this text" can be written as the pattern `\d{6}`.

Regex looks strange at first, but it is extremely useful for cleaning data and finding information in text. In Python, we use the `re` module.

In [1]:
import re

text = "Call Kolkata office at 033-2254-1234 or Siliguri at 0353-2512345. PIN 700001 / 734001."
print(re.findall(r"\d+", text))          # All groups of digits

['033', '2254', '1234', '0353', '2512345', '700001', '734001']


Always write regex patterns as **raw strings** (`r"..."`), so that backslashes work correctly.

## **The Building Blocks**
| Pattern | Matches |
|---|---|
| `.` | Any one character (except new line) |
| `\d` | A digit (0-9) |
| `\D` | Not a digit |
| `\w` | A word character (letter, digit, `_`) |
| `\s` | A space, tab or new line |
| `[abc]` | Any one of a, b or c |
| `[^abc]` | Anything except a, b, c |
| `[a-z]` | Any small letter from a to z |
| `^` | Start of the text |
| `$` | End of the text |
| `\b` | Edge of a word |

## **How Many Times (Quantifiers)**
| Pattern | Meaning |
|---|---|
| `*` | 0 or more times |
| `+` | 1 or more times |
| `?` | 0 or 1 time |
| `{n}` | Exactly n times |
| `{n,m}` | Between n and m times |

In [2]:
print(re.findall(r"\d{6}", text))                 # Exactly 6 digits: the PIN codes
print(re.findall(r"\b[A-Z]\w+", text))             # Words starting with a capital letter
print(re.findall(r"\d{3,4}-\d+(?:-\d+)?", text))   # Phone numbers

['251234', '700001', '734001']
['Call', 'Kolkata', 'Siliguri', 'PIN']
['033-2254-1234', '0353-2512345']


## **The Main Functions of re**
| Function | What it does |
|---|---|
| `re.search()` | Finds the **first** match anywhere |
| `re.match()` | Checks only at the **start** |
| `re.fullmatch()` | The **whole** text must match |
| `re.findall()` | Returns a **list** of all matches |
| `re.finditer()` | Gives the matches one by one, with their positions |
| `re.sub()` | **Replaces** matches |
| `re.split()` | **Splits** the text at the matches |

In [3]:
m = re.search(r"\d{6}", text)
print(m)
print(m.group(), m.start(), m.end())

print(re.match(r"\d+", text))                         # None: the text does not start with a digit
print(re.fullmatch(r"\d{6}", "700001"))               # Matches
print(re.fullmatch(r"\d{6}", "7000011"))              # None: 7 digits

<re.Match object; span=(57, 63), match='251234'>
251234 57 63
None
<re.Match object; span=(0, 6), match='700001'>
None


In [4]:
print(re.sub(r"\d", "#", "OTP is 482913"))
print(re.split(r"[,;|\s]+", "Kolkata, Howrah;Hooghly | Nadia   Bankura"))

OTP is ######
['Kolkata', 'Howrah', 'Hooghly', 'Nadia', 'Bankura']


## **Groups ( )**
Round brackets `( )` in a pattern **capture** parts of the match, so we can take them out separately.

In [5]:
dates = "Exam on 10/02/2025, result on 15/05/2025"

for d, m, y in re.findall(r"(\d{2})/(\d{2})/(\d{4})", dates):
    print(f"{y}-{m}-{d}")

# Rearrange with sub: \1, \2, \3 mean group 1, 2, 3
print(re.sub(r"(\d{2})/(\d{2})/(\d{4})", r"\3-\2-\1", dates))

2025-02-10
2025-05-15
Exam on 2025-02-10, result on 2025-05-15


## **Example from Remote Sensing: Finding Scene IDs in a Folder List**
When we download satellite data, we get many files. Regex helps us pick out the right ones and read the information from their names.

In [6]:
files = """
LC08_L1TP_138044_20231110_20231115_02_T1_B4.TIF
LC08_L1TP_138044_20231110_20231115_02_T1_B5.TIF
LC08_L1TP_138045_20240125_20240201_02_T1_B4.TIF
S2A_MSIL2A_20240105T045141_N0510_R076_T45QXF_20240105T083015.zip
readme.txt
S2B_MSIL2A_20240120T044929_N0510_R076_T45QXF_20240120T071122.zip
notes_kolkata.docx
"""

landsat = re.findall(r"LC08_\w{4}_(\d{3})(\d{3})_(\d{8})_\w+_B(\d+)\.TIF", files)
for path, row, date, band in landsat:
    print(f"Landsat 8  path {path} row {row}  date {date}  band {band}")

sentinel = re.findall(r"(S2[AB])_MSIL2A_(\d{8})T\d{6}_\w+_T(\w{5})_", files)
for sat, date, tile in sentinel:
    print(f"Sentinel-2 {sat}  date {date}  tile {tile}")

Landsat 8  path 138 row 044  date 20231110  band 4
Landsat 8  path 138 row 044  date 20231110  band 5
Landsat 8  path 138 row 045  date 20240125  band 4
Sentinel-2 S2A  date 20240105  tile 45QXF
Sentinel-2 S2B  date 20240120  tile 45QXF


Let us understand the first pattern:
- `LC08_` matches the text exactly.
- `\w{4}_` matches the 4-letter processing level (like L1TP) and an underscore.
- `(\d{3})(\d{3})` captures the path and the row as two groups.
- `(\d{8})` captures the 8-digit date.
- `B(\d+)\.TIF` captures the band number. `\.` means a real dot.

## **Flags: Ignore Case and Multi-Line**

In [7]:
s = "KOLKATA kolkata Kolkata"
print(re.findall(r"kolkata", s))
print(re.findall(r"kolkata", s, flags=re.IGNORECASE))

lines = "Howrah\nSealdah\nShalimar"
print(re.findall(r"^S\w+", lines, flags=re.MULTILINE))      # ^ works at the start of each line

['kolkata']
['KOLKATA', 'kolkata', 'Kolkata']
['Sealdah', 'Shalimar']


## **Greedy vs Lazy**
`*` and `+` try to match **as much as possible** (greedy). Adding `?` makes them match **as little as possible** (lazy).

In [8]:
html = "<b>Howrah</b> and <b>Sealdah</b>"
print(re.findall(r"<b>.*</b>", html))       # Greedy: one big match
print(re.findall(r"<b>(.*?)</b>", html))    # Lazy: two small matches

['<b>Howrah</b> and <b>Sealdah</b>']
['Howrah', 'Sealdah']


## **compile(): Reusing a Pattern**

In [9]:
wb_pin = re.compile(r"\b7[0-4]\d{4}\b")
for s in ["Kolkata 700091", "Delhi 110001", "Kharagpur 721302", "Siliguri 734001"]:
    m = wb_pin.search(s)
    print(f"{s:<18} -> {'WB PIN ' + m.group() if m else 'not WB'}")

Kolkata 700091     -> WB PIN 700091
Delhi 110001       -> not WB
Kharagpur 721302   -> WB PIN 721302
Siliguri 734001    -> WB PIN 734001


In [10]:
# Quick question
# Guess the output of this code cell
# import re
# print(re.findall(r"\d+", "a1b22c333"))